In [4]:
import json

from datasets import load_dataset
from tqdm import tqdm
from vllm import LLM, SamplingParams

from code_prompt.prompts.sst import NL_PROMPT, CODE_PROMPT
from code_prompt.models.lmstudio import get_response

ds = load_dataset("stanfordnlp/sst2", split="validation")
sampling_params = SamplingParams(temperature=0)
llm = LLM(model="hugging-quants/Llama-3.2-1B-Instruct-Q8_0-GGUF")

ValidationError: 1 validation error for ModelConfig
  Value error, Invalid repository ID or local directory specified: 'hugging-quants/Llama-3.2-1B-Instruct-Q8_0-GGUF'.
Please verify the following requirements:
1. Provide a valid Hugging Face repository ID.
2. Specify a local directory that contains a recognized configuration file.
   - For Hugging Face models: ensure the presence of a 'config.json'.
   - For Mistral models: ensure the presence of a 'params.json'.
3. For GGUF: pass the local path of the GGUF checkpoint.
   Loading GGUF from a remote repo directly is not yet supported.
 [type=value_error, input_value=ArgsKwargs((), {'model': ..., 'model_impl': 'auto'}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

In [ ]:
examples = []
for i in tqdm(range(50)):
    text = ds[i]["sentence"]
    nl_prompt = NL_PROMPT.format(text=text)
    response = get_response(nl_prompt, model="Qwen/Qwen2.5-14B-Instruct-GGUF")
    examples.append(
        {
            "id": ds[i]["idx"],
            "text": text,
            "label": ds[i]["label"],
            "nl_response": response,
        }
    )
with open("sst2_examples.json", "w") as f:
    json.dump(examples, f, indent=4, ensure_ascii=True)

In [ ]:
hit = 0
for example in examples:
    if "POSITIVE" in example["nl_response"][:20]:
        example["pred"] = 1
    elif "NEGATIVE" in example["nl_response"][:20]:
        example["pred"] = 0
    else:
        example["pred"] = -1
    if example["pred"] == example["label"]:
        hit += 1
print(f"Accuracy: {hit / len(examples) * 100:.2f}%")
# save predictions to json
with open("sst2_predictions.json", "w") as f:
    json.dump(examples, f, indent=4, ensure_ascii=True)

Accuracy: 96.00%


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

examples = []
TOKENIZER = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-14B")
MODEL = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-14B", device_map="auto").eval()
DEVICE = "cuda"

for i in tqdm(range(50)):
    text = ds[i]["sentence"]
    code_prompt = CODE_PROMPT.format(text=text)
    model_inputs = TOKENIZER([code_prompt], return_tensors="pt").to(DEVICE)
    generated_ids = MODEL.generate(model_inputs.input_ids, max_new_tokens=512, do_sample=False)[0]
    response = TOKENIZER.decode(generated_ids[len(model_inputs.input_ids[0]):], skip_special_tokens=True)
    examples.append(
        {
            "id": ds[i]["idx"],
            "text": text,
            "label": ds[i]["label"],
            "code_response": response,
        }
    )
# save to file
with open("sst2_code_examples.json", "w") as f:
    json.dump(examples, f, indent=4, ensure_ascii=True)

c:\Users\yzche\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yzche\.cache\huggingface\hub\models--Qwen--Qwen2.5-Coder-14B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading checkpoint shards: 100%|██████████| 6/6 [00:10<00:00,  1.77s/it]
Some parameters ar